# BIOMASS TomoSAR — Original data loader examples

This notebook implements the examples in `BIOMASS_LOADER_README.md` using the **original, unmodified** `biomass_tomosar.bm_data_loader`.

**What you'll do:** discover the seven product directories; identify the **fourth chronological acquisition** as master; check reference annotation geometry; initialize `bmStack`; inspect NetCDF lookup tables; estimate memory; read small complex-SLC tiles with GDAL; display amplitude and a wrapped interferogram. Full-resolution LUT interpolation and full-stack SLC import are **disabled by default**.

**Before you start:** open this repository through **VS Code Remote–WSL**, select the project `.venv/bin/python` notebook kernel, and set `config/local.toml`:

```toml
[paths]
biomass_stack = "data/BIOMASS_STACK"
```

You don't need to run the Mondah reference notebook or install `ste_io`.

**Terminology:** chronological **index 3** is the physical master acquisition. `bmStack` separately uses `baseDirs[0]` as the *source of primary geometry*, which doesn't necessarily identify which SLC is the master.

## 1. Imports and data location (no SLC imagery loaded)

In [ ]:
import os
print("Kernel PID:", os.getpid())

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from lxml import etree
from osgeo import gdal, gdal_array  # gdal_array checks GDAL's NumPy extension

from biomass_tomosar.paths import biomass_stack_dir
from biomass_tomosar.bm_data_loader import bmStack

root = biomass_stack_dir()
print("Dataset:", root)
print("GDAL:", gdal.VersionInfo("--version"))
print("NumPy:", np.__version__)
if not root.is_dir():
    raise FileNotFoundError(f"Data folder not found: {root}. Check config/local.toml")

## 2. Discover the seven products and validate expected files

The original loader builds filenames from the first **70 characters** of each product directory name. Don't rename BIOMASS product folders or files. Sorting the supplied names is expected to give chronological order for the `BIO_S2_STA...YYYYMMDD...` product names; **inspect the printed list** before assuming that it does.

All seven acquisitions reportedly have `annotation_primary/`. That directory's presence **does not** identify the master.

In [ ]:
products = sorted(p for p in root.glob("BIO*") if p.is_dir())

print(f"Found {len(products)} acquisition folders:")
for i, product in enumerate(products):
    print(f"  [{i}] {product.name}" + ("  <-- 4TH / MASTER" if i == 3 else ""))

if len(products) != 7:
    raise ValueError(f"Expected 7 BIOMASS acquisitions, found {len(products)}. Check the data path.")

chronological_master_idx = 3
master_product = products[chronological_master_idx]


def files_for(product):
    prefix = product.name[:70].lower()
    return {
        "primary XML": product / "annotation_primary" / f"{prefix}_annot.xml",
        "coreg XML": product / "annotation_coregistered" / f"{prefix}_annot.xml",
        "coreg LUT": product / "annotation_coregistered" / f"{prefix}_lut.nc",
        "amplitude TIFF": product / "measurement" / f"{prefix}_i_abs.tiff",
        "phase TIFF": product / "measurement" / f"{prefix}_i_phase.tiff",
    }

missing = []
for i, product in enumerate(products):
    present = files_for(product)
    print(f"\n[{i}] {product.name}")
    for label, path in present.items():
        exists = path.is_file()
        print(f"  {label:<16} {exists}")
        if not exists:
            missing.append(str(path))

if missing:
    raise FileNotFoundError("Missing expected files:\n" + "\n".join(missing))

## 3. Compare primary-annotation geometry

The loader parses primary-annotation XML from `baseDirs[0]`. Since **every product has that folder**, the question is whether they describe the **same primary grid**. The following compares geometry fields actually used by the loader; it is *not* a complete validation of the scientific co-registration or reference selection.

If all grids match, keep chronological ordering and use master index 3. If they differ (or essential fields are unavailable), place the known master folder first for the loader and explicitly retain a **loader-index → chronological-index** mapping.

In [ ]:
PRIMARY_GRID_FIELDS = (
    "firstSampleSlantRangeTime",
    "rangeTimeInterval",
    "numberOfSamples",
    "firstLineAzimuthTime",
    "azimuthTimeInterval",
    "azimuthPixelSpacing",
    "numberOfLines",
    "noDataValue",
)


def primary_grid(product):
    xml = files_for(product)["primary XML"]
    tree = etree.parse(str(xml))
    sar_nodes = tree.xpath("sarImage")
    if not sar_nodes:
        raise ValueError(f"No sarImage element in {xml}")
    sar = sar_nodes[0]
    return {name: sar.findtext(name) for name in PRIMARY_GRID_FIELDS}


primary_grids = [primary_grid(p) for p in products]
master_grid = primary_grids[chronological_master_idx]
valid_fields = all(value is not None for grid in primary_grids for value in grid.values())
all_grids_equal = valid_fields and all(grid == master_grid for grid in primary_grids)

for i, grid in enumerate(primary_grids):
    differences = [name for name in PRIMARY_GRID_FIELDS if grid[name] != master_grid[name]]
    status = "matches master" if not differences else f"differs in {differences}"
    print(f"[{i}] {status}")

if all_grids_equal:
    ordered_products = products.copy()
    print("\nPrimary geometry matches: keeping chronological stack order.")
else:
    ordered_products = [master_product] + [p for p in products if p != master_product]
    print("\nPrimary grids differ or are incomplete: putting known master first for bmStack.")
    print("Review the primary annotations and product documentation before interpreting geometry.")

# Retain index mapping for subsequent interferogram choices.
loader_to_chronological = [products.index(p) for p in ordered_products]
master_idx = ordered_products.index(master_product)
base_dirs = [str(p) for p in ordered_products]  # bmStack expects strings

print("\nLoader index -> chronological index:")
for loader_idx, chron_idx in enumerate(loader_to_chronological):
    print(f"  loader {loader_idx}: chronological {chron_idx}" +
          ("  <-- MASTER" if loader_idx == master_idx else ""))

## 4. Initialize `bmStack` — metadata and LUT handles only

`bmStack(base_dirs)` parses annotation XML, opens one NetCDF LUT per acquisition, computes the wavelength with a hard-coded carrier frequency of **435 MHz**, and constructs range/azimuth coordinate arrays. It **does not** call `importSlcs()` or load the full TIFF images.

The original loader uses a fixed time reference of `2025-05-22T09:47:44` for azimuth times. Do not treat `stack.t` as Unix epoch time.

In [ ]:
stack = bmStack(base_dirs)

if not stack._isStack:
    raise RuntimeError("The loader fell back to non-stack mode; check annotation paths.")

print("Acquisitions:", stack.Ntracks)
print("Radar wavelength (m):", stack.lamb)
print("Lines (azimuth):", stack.nli)
print("Samples (range):", stack.nsa)
print("Slant-range extent (km):", stack.rg[0] / 1e3, "to", stack.rg[-1] / 1e3)
print("Azimuth extent (km):", stack.az[0] / 1e3, "to", stack.az[-1] / 1e3)
print("LUT objects:", len(stack.luts))
print("Master loader index:", master_idx)

# Access the master geometry XML tree or an individual co-registered annotation.
print("Primary annotation type:", type(stack.annot_prim).__name__)
print("Master co-registered annotation type:", type(stack.annot_coreg[master_idx]).__name__)

## 5. Explore NetCDF lookup tables (`bmLut` / `lutWrapper`)

For one `stack.luts[i]`:

- `.get().nc` — underlying NetCDF dataset.
- `.get("group").nc` — NetCDF group.
- `.get("group", "variable").data` — native-resolution NumPy data.
- `.get("group", "variable").upsampled` — **full-size** interpolation, memory intensive (optional below).

The loader interpolates LUTs by **azimuth time** and **slant-range time** using linear interpolation. The meaning/sign convention of a LUT variable must still come from the BIOMASS product documentation.

In [ ]:
lut = stack.luts[master_idx]
dataset = lut.get().nc
print("Top-level groups:", list(dataset.groups))
print("Top-level variables:", list(dataset.variables))

coreg = lut.get("coregistration").nc
print("Coregistration LUT variables:", list(coreg.variables))

flat_native = lut.get("coregistration", "flatteningPhaseScreen").data
print("Native flattening-phase LUT shape:", flat_native.shape)
print("Native LUT dtype:", flat_native.dtype)

plt.figure(figsize=(7, 4))
plt.imshow(flat_native, cmap="twilight")
plt.colorbar(label="Stored flattening phase (check product units)")
plt.title("Master acquisition — native-resolution flattening LUT")
plt.xlabel("LUT range index")
plt.ylabel("LUT azimuth index")
plt.show()

### Optional: interpolate one LUT onto the full SAR image

This is **not** performed automatically. `.upsampled` creates large coordinate arrays and can use considerably more RAM than the output alone. Keep `RUN_FULL_RES_LUT = False` for initial exploration.

In [ ]:
RUN_FULL_RES_LUT = False

estimated_phase_gib = stack.nli * stack.nsa * 8 / (1024**3)  # float64 output estimate
print(f"Approx. full-size phase array alone: {estimated_phase_gib:.2f} GiB")

if RUN_FULL_RES_LUT:
    full_flat_phase = lut.get("coregistration", "flatteningPhaseScreen").upsampled
    print("Upsampled LUT shape:", full_flat_phase.shape)
else:
    print("Skipped full-resolution interpolation. Set RUN_FULL_RES_LUT=True to run.")

## 6. Inspect TIFF metadata and estimate RAM

TIFF band order in the source example is **HH, HV, VH, VV**, but confirm that this matches your products before assigning polarization names. For co-registered products, the final full SLC array is intended to be `(acquisitions, polarizations, azimuth lines, range samples)` with `swapaxes=False`.

A `complex64` sample uses **8 bytes**. The estimate below excludes temporary amplitude/phase arrays, exponentials, and flattening LUTs, so actual peak memory can be much higher.

In [ ]:
def describe_tiffs(product):
    paths = files_for(product)
    results = {}
    for key in ("amplitude TIFF", "phase TIFF"):
        ds = gdal.Open(str(paths[key]), gdal.GA_ReadOnly)
        if ds is None:
            raise OSError(f"GDAL could not open {paths[key]}")
        results[key] = (ds.RasterCount, ds.RasterYSize, ds.RasterXSize)
        print(f"{key}: bands={ds.RasterCount}, rows={ds.RasterYSize}, columns={ds.RasterXSize}, "
              f"band-1-type={gdal.GetDataTypeName(ds.GetRasterBand(1).DataType)}")
        ds = None
    if results["amplitude TIFF"] != results["phase TIFF"]:
        raise ValueError("Amplitude and phase TIFF shapes differ")
    return results["amplitude TIFF"]


n_pol, n_az, n_rg = describe_tiffs(ordered_products[master_idx])
final_gib = stack.Ntracks * n_pol * n_az * n_rg * np.dtype(np.complex64).itemsize / (1024**3)
print(f"Final complex64 SLC stack alone: ~{final_gib:.2f} GiB")
print("Expected annotation dimensions:", stack.nli, stack.nsa)
if (n_az, n_rg) != (stack.nli, stack.nsa):
    print("WARNING: TIFF row/column dimensions differ from primary geometry; verify axes/layout.")

## 7. Read just a small complex-SLC window

`bmStack.importSlcs()` cannot read a region: it loads *all* products. This helper **does not modify the original loader**; it uses the same two TIFFs and reconstructs complex samples as `amplitude * exp(1j * phase)`.

The helper deliberately **does not apply** the `flatteningPhaseScreen`, so initial plots and the interferogram below show unflattened phase. The column offset `xoff` is range/sample, row offset `yoff` is azimuth/line.

In [ ]:
def read_slc_window(stack, track, xoff, yoff, width=512, height=512):
    if not 0 <= track < stack.Ntracks:
        raise IndexError(f"Track {track} outside [0, {stack.Ntracks - 1}]")

    product = Path(stack.baseDirs[track])
    paths = files_for(product)
    amp_ds = gdal.Open(str(paths["amplitude TIFF"]), gdal.GA_ReadOnly)
    phase_ds = gdal.Open(str(paths["phase TIFF"]), gdal.GA_ReadOnly)

    if amp_ds is None or phase_ds is None:
        raise FileNotFoundError(f"Cannot open amplitude and/or phase TIFFs for {product}")
    if (amp_ds.RasterCount, amp_ds.RasterXSize, amp_ds.RasterYSize) != (
        phase_ds.RasterCount, phase_ds.RasterXSize, phase_ds.RasterYSize
    ):
        raise ValueError("Amplitude/phase TIFFs are not the same dimensions")
    if min(xoff, yoff) < 0 or min(width, height) <= 0:
        raise ValueError("Offsets must be >= 0 and dimensions > 0")
    if xoff + width > amp_ds.RasterXSize or yoff + height > amp_ds.RasterYSize:
        raise ValueError("Requested window exceeds TIFF dimensions")

    amplitude = amp_ds.ReadAsArray(xoff, yoff, width, height)
    phase = phase_ds.ReadAsArray(xoff, yoff, width, height)
    no_data = stack.noDataValue if stack._isStack else stack.noDataValue[track]

    # Original loader's handling of no-data samples.
    amplitude = np.where(amplitude == no_data, 0, amplitude)
    phase = np.where(phase == no_data, 0, phase)
    slc = (amplitude * np.exp(1j * phase)).astype(np.complex64)
    amp_ds = phase_ds = None
    return slc


TILE_SIZE = min(512, n_az, n_rg)
xoff = (n_rg - TILE_SIZE) // 2
yoff = (n_az - TILE_SIZE) // 2

master_tile = read_slc_window(stack, master_idx, xoff, yoff, TILE_SIZE, TILE_SIZE)
print("Master tile shape (bands, azimuth, range):", master_tile.shape)
print("dtype:", master_tile.dtype)

# First band is expected HH according to the author's original example.
master_first_band = master_tile[0] if master_tile.ndim == 3 else master_tile
amplitude_db = 20 * np.log10(np.maximum(np.abs(master_first_band), 1e-8))

plt.figure(figsize=(7, 6))
plt.imshow(amplitude_db, cmap="gray")
plt.colorbar(label="Amplitude (dB, uncalibrated)")
plt.title(f"Master (chronological #4) — first polarization band, {TILE_SIZE}x{TILE_SIZE}")
plt.xlabel("Range pixel within tile")
plt.ylabel("Azimuth pixel within tile")
plt.show()

### Optional: visualize all four polarizations in the master window

In [ ]:
if master_tile.ndim == 3 and master_tile.shape[0] == 4:
    assumed_labels = ["HH", "HV", "VH", "VV"]  # confirm band order against product metadata
    fig, axes = plt.subplots(2, 2, figsize=(10, 8), layout="constrained")
    for ax, pol, label in zip(axes.flat, master_tile, assumed_labels):
        ax.imshow(20 * np.log10(np.maximum(np.abs(pol), 1e-8)), cmap="gray", vmin=-30, vmax=10)
        ax.set_title(f"Band: {label} (assumed)")
        ax.set_xlabel("Range pixel")
        ax.set_ylabel("Azimuth pixel")
    plt.show()
else:
    print("Expected a four-band TIFF to display HH/HV/VH/VV; inspect actual band layout.")

## 8. Make a small wrapped interferogram against the master

For co-registered master SLC `S_m` and another acquisition `S_s`, compute

\[I = S_m\,S_s^*,\qquad \phi = \arg(I).\]

The product order controls the phase sign. Here we choose the **chronologically fifth acquisition** (index 4) as the other track if available, using the explicit loader-index mapping built earlier. Both tiles use the *same pixel window*.

**Scientific caution:** this is a raw wrapped interferogram from **unflattened** SLC data. It is *not* a height map or TomoSAR reconstruction; baseline effects, co-registration quality, SKPD/flattening, ionosphere and other phase terms require further work.

In [ ]:
other_chronological_idx = 4  # fifth acquisition; adjust if desired
other_idx = loader_to_chronological.index(other_chronological_idx)

other_tile = read_slc_window(stack, other_idx, xoff, yoff, TILE_SIZE, TILE_SIZE)
if master_tile.shape != other_tile.shape:
    raise ValueError("Master and secondary tiles have different shapes")

POL_BAND = 0  # assumed HH, pending confirmation of band order
sm = master_tile[POL_BAND] if master_tile.ndim == 3 else master_tile
ss = other_tile[POL_BAND] if other_tile.ndim == 3 else other_tile

interferogram = sm * np.conj(ss)
wrapped_phase = np.angle(interferogram)
invalid = (np.abs(sm) == 0) | (np.abs(ss) == 0)
wrapped_phase = np.where(invalid, np.nan, wrapped_phase)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
axes[0].imshow(20 * np.log10(np.maximum(np.abs(ss), 1e-8)), cmap="gray")
axes[0].set_title("Secondary — first polarization amplitude (dB)")
plot = axes[1].imshow(wrapped_phase, cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[1].set_title("Wrapped phase: master × conj(secondary)")
fig.colorbar(plot, ax=axes[1], label="Phase (radians)")
for ax in axes:
    ax.set_xlabel("Range pixel within tile")
    ax.set_ylabel("Azimuth pixel within tile")
plt.show()

## 9. Optional: import the entire stack with the original loader

**Disabled by default.** Only enable this if the expected final `complex64` array **and all temporary arrays** fit comfortably in WSL RAM. The original `importSlcs()` loads **all acquisitions and all polarizations**, and with `terrain_flatten=True` it also interpolates large phase screens. It has no region argument.

- `terrain_flatten=False` preserves the phase read from TIFFs (after complex reconstruction), like our tile helper.
- `swapaxes=False` keeps arrays in `(acquisition, polarization, azimuth, range)` order.
- The original method appends to `_slcs`: **do not call it twice on the same instance**. This optional cell constructs a fresh instance.

In [ ]:
RUN_FULL_STACK = False
MAX_FINAL_STACK_GIB = 2.0  # safety guard, adjust only after checking available memory

if RUN_FULL_STACK:
    if final_gib > MAX_FINAL_STACK_GIB:
        raise MemoryError(
            f"Final array estimated {final_gib:.2f} GiB, above your {MAX_FINAL_STACK_GIB:.2f} GiB guard. "
            "Peak usage can be several times larger."
        )

    full_stack = bmStack(base_dirs)  # fresh object: importSlcs appends to its internal list
    full_stack.importSlcs(terrain_flatten=False, swapaxes=False)
    print("All SLCs shape:", full_stack.slcs.shape)
    print("All SLCs dtype:", full_stack.slcs.dtype)
    print("Master index in full_stack.slcs:", master_idx)
else:
    print("Full stack import skipped. The earlier windowed examples are sufficient for first exploration.")

## 10. Next steps and caveats

- Verify that the selected seventh-product stack and its primary annotations correspond to the intended reference. Having `annotation_primary/` on all acquisitions **does not** establish master identity; your known master remains **chronological acquisition #4**.
- Confirm the TIFF **band order** and units from the BIOMASS product-format documentation.
- Examine the `flatteningPhaseScreen`, baseline/vertical-wavenumber geometry, and relevant **SKPD phase corrections** before interpreting phase differences physically.
- Estimate coherence using multilooking/windowing before attempting TomoSAR inversion.
- The original loader hard-codes a time origin and frequency and does **not** calculate baselines, coherence, or tomography.

**Original implementation:** `src/biomass_tomosar/bm_data_loader.py` (unmodified). **Detailed explanations:** `BIOMASS_LOADER_README.md`. The notebook has been syntax-checked but **not executed against your private SAR data**.